# AegisPlan: Fast QLoRA Fine-Tuning on Google Colab
**Autonomous Neuro-Symbolic Healthcare Spatial Synthesis Engine**

This notebook fine-tunes an LLM to generate code-compliant `FML-DSL` clinical floorplans using standard Hugging Face PEFT + TRL + BitsAndBytes (4-bit NF4).
- **Install time**: ~20 seconds (uses official pre-built binary wheels)
- **Training time**: ~12 to 15 minutes on a free Google Colab **T4 GPU**
- **Hardware**: Works on free T4 GPU (16GB VRAM) or A100 / L4 on Colab Pro

### Step 1: Fast Install (Takes ~20 Seconds)

In [ ]:
!pip install -q -U trl peft bitsandbytes datasets transformers accelerate

### Step 2: Load Model in 4-bit NF4 with LoRA ($r=32, \alpha=64$)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Qwen2.5-7B-Instruct has outstanding structured code and CAD syntax adherence on T4
model_id = "Qwen/Qwen2.5-7B-Instruct"

print(f"Loading base model: {model_id} in 4-bit NF4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# LoRA Configuration per Section 5 of AegisPlan Master Plan
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

### Step 3: Load Certified Dataset (`train.jsonl` and `val.jsonl`)
Drag & drop `train.jsonl` and `val.jsonl` from your machine into Colab's left-hand file browser (`📁 Files`).

In [ ]:
import os
from datasets import load_dataset

if not os.path.exists("train.jsonl"):
    raise FileNotFoundError("Please upload train.jsonl & val.jsonl using Colab's left files panel!")

train_dataset = load_dataset("json", data_files="train.jsonl", split="train")
val_dataset = load_dataset("json", data_files="val.jsonl", split="train")
print(f"Loaded verified train split: {len(train_dataset)} samples")
print(f"Loaded verified val split:   {len(val_dataset)} samples")

def format_prompts(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False) for m in batch["messages"]]}

train_data = train_dataset.map(format_prompts, batched=True)
val_data = val_dataset.map(format_prompts, batched=True)
print("Sample chat formatted prompt preview:")
print(train_data[0]["text"][:350] + "...")

### Step 4: Run Fine-Tuning (~12–15 Minutes on Colab T4)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./aegisplan_adapter",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    dataset_text_field="text",
    max_seq_length=1536,
    args=training_args,
)

print("Beginning QLoRA training...")
trainer.train()
print("Training completed successfully!")

### Step 5: Test Generation on Held-Out Test Brief

In [ ]:
import json

# Check if test.jsonl was uploaded, otherwise use standard benchmark prompt
if os.path.exists("test.jsonl"):
    with open("test.jsonl", "r") as f:
        test_sample = json.loads(f.readline())
    test_messages = test_sample["messages"][:2]
    print("Testing on held-out sample:", test_messages[1]["content"][:120], "...")
else:
    test_messages = [
        {"role": "system", "content": "You are AegisPlan, an autonomous AI architect specializing in U.S. Healthcare Outpatient Facilities.\nOutput ONLY valid FML-DSL code matching the exact grammar: BOUNDARY RECT 0 0 <W> <H>, ROOM, DOOR, FDOOR, END_PLAN."},
        {"role": "user", "content": "Generate an outpatient clinic floorplan for boundary RECT 0 0 16000 12000.\nClinical Brief: Design a 16m x 12m Primary care outpatient clinic requiring central circulation corridor, 3 outpatient exam rooms (>= 11.15 m2 each), single-user ADA compliant restroom, clean and soiled utility support rooms.\nEmit strictly valid FML-DSL:"}
    ]

inputs = tokenizer.apply_chat_template(test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.7, top_p=0.9)

generated_fml = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("\n=== GENERATED FML-DSL CLINICAL FLOORPLAN ===")
print(generated_fml)

### Step 6: Save & Download Adapter Zip

In [ ]:
# Save adapter weights
output_dir = "./aegisplan_final_lora"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Saved LoRA adapter to {output_dir}")

# Create zip archive for 1-click download
!zip -r aegisplan_final_lora.zip aegisplan_final_lora

from google.colab import files
files.download("aegisplan_final_lora.zip")
print("Download started! Extract this into your AegisPlan/checkpoints/ directory.")